# Descrição geral
Os dados foram sintetisados usando um script simples, aqui o foco é gerar um conjunto mais simples, mais rapido e mais facil de usar no treinamento

In [ ]:
from geradores import gera_dados_perfil_usuario
df_perfil_usuario = gera_dados_perfil_usuario.run(5000)

In [ ]:
print('Trechos')
print(df_perfil_usuario.head())

print('Shape')
print(df_perfil_usuario.shape)

print('Potenciais falhas')
print(df_perfil_usuario.isnull().sum())

Adiciona colunas relativas ao usuario


In [ ]:
colunas_gasto = df_perfil_usuario.filter(like="gasto_").columns
df_perfil_usuario['total_gasto'] = df_perfil_usuario[colunas_gasto].sum(axis=1)

df_perfil_usuario['percentual_gasto'] = df_perfil_usuario['total_gasto'] / df_perfil_usuario['renda_mensal']
df_perfil_usuario['percentual_investido'] = df_perfil_usuario['valor_investido'] / df_perfil_usuario['renda_mensal']
df_perfil_usuario['saldo'] = df_perfil_usuario['renda_mensal'] - df_perfil_usuario['total_gasto'] - df_perfil_usuario['valor_investido']

Definir perfil do usuário

In [ ]:
def definir_perfil(linha):
    if linha['saldo'] < 0 or linha['percentual_gasto'] > 0.9:
        return "Em risco"
    elif linha['percentual_gasto'] > 0.7 or linha['percentual_investido'] < 0.05:
        return "Em observacao"
    else:
        return "Saudavel"

df_perfil_usuario['perfil_financeiro'] = df_perfil_usuario.apply(definir_perfil, axis=1)

Definir valores

In [ ]:
x = df_perfil_usuario.drop(columns=['perfil_financeiro'])
y = df_perfil_usuario['perfil_financeiro']

Treinar modelo

In [ ]:
from sklearn.model_selection import train_test_split

x_treino, x_teste, y_treino, y_teste = train_test_split(
    x, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Tamanho treino:", x_treino.shape)
print("Tamanho teste:", x_teste.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

modelo_perfil = RandomForestClassifier(random_state=42)
modelo_perfil.fit(x_treino, y_treino)

Metricas

In [ ]:
from sklearn.metrics import classification_report

y_previsto = modelo_perfil.predict(x_teste)
print(classification_report(y_teste, y_previsto))

Testes 

In [ ]:
import pandas as pd

casos_teste = pd.DataFrame([
    # Caso 1: gasto exatamente no limite de "Em risco" (percentual_gasto ~ 0.9)
    {
        "renda_mensal": 5000, "valor_investido": 200,
        "gasto_alimentação": 800, "gasto_transporte": 500, "gasto_saúde": 400,
        "gasto_moradia": 1200, "gasto_educação": 300, "gasto_lazer": 200,
        "gasto_serviços": 200, "gasto_assinaturas": 100, "gasto_dívidas": 500,
        "gasto_outras": 300,
    },
    # Caso 2: saldo bem próximo de zero (mas positivo)
    {
        "renda_mensal": 5000, "valor_investido": 500,
        "gasto_alimentação": 700, "gasto_transporte": 400, "gasto_saúde": 300,
        "gasto_moradia": 1000, "gasto_educação": 300, "gasto_lazer": 300,
        "gasto_serviços": 300, "gasto_assinaturas": 100, "gasto_dívidas": 600,
        "gasto_outras": 490,
    },
    # Caso 3: investimento exatamente no limite de 5% da renda
    {
        "renda_mensal": 6000, "valor_investido": 300,  # exatamente 5%
        "gasto_alimentação": 500, "gasto_transporte": 300, "gasto_saúde": 300,
        "gasto_moradia": 900, "gasto_educação": 200, "gasto_lazer": 200,
        "gasto_serviços": 200, "gasto_assinaturas": 100, "gasto_dívidas": 300,
        "gasto_outras": 200,
    },
    # Caso 4: perfil claramente saudável (referência, sem ambiguidade)
    {
        "renda_mensal": 10000, "valor_investido": 3000,
        "gasto_alimentação": 500, "gasto_transporte": 300, "gasto_saúde": 300,
        "gasto_moradia": 1000, "gasto_educação": 200, "gasto_lazer": 200,
        "gasto_serviços": 100, "gasto_assinaturas": 50, "gasto_dívidas": 100,
        "gasto_outras": 100,
    },
    # Caso 5: perfil claramente em risco (referência, sem ambiguidade)
    {
        "renda_mensal": 3000, "valor_investido": 0,
        "gasto_alimentação": 600, "gasto_transporte": 400, "gasto_saúde": 300,
        "gasto_moradia": 1200, "gasto_educação": 200, "gasto_lazer": 300,
        "gasto_serviços": 200, "gasto_assinaturas": 100, "gasto_dívidas": 500,
        "gasto_outras": 300,
    },
])

# Calcular os mesmos indicadores derivados usados no treino
colunas_gasto = casos_teste.filter(like="gasto_").columns
casos_teste['total_gasto'] = casos_teste[colunas_gasto].sum(axis=1)
casos_teste['percentual_gasto'] = casos_teste['total_gasto'] / casos_teste['renda_mensal']
casos_teste['percentual_investido'] = casos_teste['valor_investido'] / casos_teste['renda_mensal']
casos_teste['saldo'] = casos_teste['renda_mensal'] - casos_teste['total_gasto'] - casos_teste['valor_investido']

# Aplicar a MESMA função de regras, pra comparar com a previsão do modelo
casos_teste['perfil_regra'] = casos_teste.apply(definir_perfil, axis=1)
print(x.columns)
print(casos_teste.columns)

# # Reordenar as colunas igual ao x de treino, e prever com o modelo
# casos_teste_x = casos_teste[x.columns]
# casos_teste['perfil_modelo'] = modelo_perfil.predict(casos_teste_x)

# print(casos_teste[['percentual_gasto', 'percentual_investido', 'saldo', 'perfil_regra', 'perfil_modelo']])

In [ ]:
import joblib

joblib.dump(modelo_perfil, "../servico-dados/modelos/modelo_perfil.joblib")
joblib.dump(list(x.columns), "../servico-dados/modelos/colunas_perfil.joblib")
